# Exploring SesamEO API for Download and Visualization

## Rotterdam Use Case

### Set your SesamEO API key
Follow instructions [here](https://platform.destine.eu/docs/sesameo/doc/index.html#api-key) to obtain an API key.

Additionally, you need to [configure your SesamEO account with the data providers](https://sesameo.destine.eu/providers) (for example CDSE, Microsoft, etc.)

You will be prompted to input your API key in the following cell.

In [ ]:
from getpass import getpass
API_KEY = getpass("SesamEO API key: ").strip()
if not API_KEY:
    raise ValueError("API key cannot be empty.")

### Import libraries

In [ ]:
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import requests
import rioxarray as rxr

### Define download function

In [ ]:
def download_file(url: str, api_key: str, output_path: Path, timeout: int = 180) -> Path:
    headers = {"X-API-KEY": api_key}
    with requests.get(url, headers=headers, stream=True, timeout=timeout) as response:
        response.raise_for_status()
        with output_path.open("wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    return output_path

### Choose a dataset to download

In [ ]:
# Global above ground biomass
# download_url = "https://api.sesameo.destine.eu/odata/v1/Collections('hgb')/Products('MICROSOFT:hgb')/Links('Global%20above-ground%20biomass')/$value"
# CCI Land Cover
download_url = "https://api.sesameo.destine.eu/odata/v1/Collections('esa-cci-lc')/Products('MICROSOFT:C3S-LC-L4-LCCS-Map-300m-P1Y-2020-v2.1.1-N45E000')/Links('DownloadLink')/$value"
file_type = "tif"
# download_url = "https://api.sesameo.destine.eu/odata/v1/Collections('esa-cci-lc')/Products('MICROSOFT:C3S-LC-L4-LCCS-Map-300m-P1Y-2020-v2.1.1-N45E000')/Links('Land%20Cover%20Class%20Defined%20in%20the%20Land%20Cover%20Classification%20System')/$value"

# set output path
output_path = Path("downloads") / f"sesameo_product.{file_type}"
output_path.parent.mkdir(parents=True, exist_ok=True)

print(f"Downloading product URL:\n {download_url}")
print(f"Product will be downloaded to:\n {output_path}")

In [ ]:
downloaded_file = download_file(download_url, API_KEY, output_path)
print(f"Downloaded: {downloaded_file}")

if downloaded_file.suffix.lower() == ".zip":
    extract_dir = output_path.parent / output_path.stem
    extract_dir.mkdir(exist_ok=True, parents=True)
    with zipfile.ZipFile(downloaded_file, "r") as zf:
        zf.extractall(extract_dir)
    print(f"Extracted to: {extract_dir}")
    all_files = sorted([p for p in extract_dir.rglob("*") if p.is_file()])
else:
    all_files = [downloaded_file]

### Preview files and open a dataset

In [ ]:
for p in all_files[:20]:
    print(p)

raster_files = [p for p in all_files if p.suffix.lower() in {".tif", ".tiff", ".nc", ".nc4"}]
if not raster_files:
    raise FileNotFoundError("No supported raster file found (.tif, .tiff, .nc, .nc4).")

dataset_path = raster_files[0]
print(f"Using dataset: {dataset_path}")

da = rxr.open_rasterio(dataset_path, masked=True)
if "band" in da.dims and da.sizes.get("band", 0) == 1:
    da = da.squeeze("band", drop=True)
da

In [ ]:
# Keep only spatial dimensions, assume lon/lat in EPSG:4326, then zoom to the Netherlands.
plot_da = da
for dim in [d for d in plot_da.dims if d not in {"lat", "lon", "y", "x", "band"}]:
    plot_da = plot_da.isel({dim: 0})

if "band" in plot_da.dims and plot_da.sizes.get("band", 0) == 1:
    plot_da = plot_da.squeeze("band", drop=True)

if "x" in plot_da.dims and "y" in plot_da.dims:
    plot_da = plot_da.rename({"x": "lon", "y": "lat"})

plot_da = plot_da.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=False)
if plot_da.rio.crs is None:
    plot_da = plot_da.rio.write_crs("EPSG:4326", inplace=False)

# Approximate Netherlands bounding box in lon/lat.
lon_min, lon_max = 3.0, 7.5
lat_min, lat_max = 50.5, 53.8

lat_slice = slice(lat_min, lat_max)
if plot_da["lat"].values[0] > plot_da["lat"].values[-1]:
    lat_slice = slice(lat_max, lat_min)

plot_nl = plot_da.sel(lon=slice(lon_min, lon_max), lat=lat_slice)
if plot_nl.size == 0:
    raise ValueError("Zoomed subset for the Netherlands is empty. Check coordinate ranges.")

plt.figure(figsize=(7, 7))
plot_nl.plot(cmap="viridis")
plt.title(f"{dataset_path.name} - Netherlands zoom")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.show()